# 02. Multi-Model Comparative Analysis

Executes scenario-based comparisons across models under identical transmitter/receiver power parameters, distances, and carrier frequencies.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from propagation.simulation.scenario import ScenarioConfig
from propagation.simulation.engine import SimulationEngine
from propagation.analysis.comparator import ModelComparator

# Load Urban Scenario
urban_cfg = ScenarioConfig.from_yaml(ROOT / 'experiments' / 'scenarios' / 'urban.yaml')
engine = SimulationEngine()
urban_sim = engine.run_distance_sweep(urban_cfg)

print(f'Simulation completed for scenario: {urban_sim.scenario_name}')


### Descriptive Statistics Summary Table


In [ ]:
stats_df = ModelComparator.calculate_summary_statistics(urban_sim)
display(stats_df) if 'display' in globals() else print(stats_df)


### Discrepancy Relative to Free Space (Delta PL = PL_model - PL_FSPL)


In [ ]:
diff_df = ModelComparator.calculate_pairwise_differences(urban_sim, baseline_model='fspl')
display(diff_df.head()) if 'display' in globals() else print(diff_df.head())


In [ ]:
plt.figure(figsize=(8, 5))
d_vals = urban_sim.sweep_values
for col in diff_df.columns:
    if col.startswith('Delta_PL'):
        plt.plot(d_vals, diff_df[col], lw=2.0, label=col.replace('Delta_PL_', '').replace('_dB', ''))
plt.xlabel('Distance (km)')
plt.ylabel('Excess Path Loss over Free Space (dB)')
plt.title('Urban Clutter Excess Path Loss Discrepancy')
plt.xscale('log')
plt.grid(True, which='both', ls=':')
plt.legend()
plt.show()
